# YZM 513 — Week 8 Lab: Your First CUDA Kernels

**Post-midterm schedule:** original syllabus Week 6 → **Week 8**.

By the end, you will be able to launch a grid, calculate thread indices, move data explicitly, synchronize, verify results, and measure a simple fusion.

**Rules for every kernel:** compute the global index → bounds-check → do one well-defined piece of work → synchronize → verify against a CPU result.

## 0. Setup

In Colab, select **Runtime → Change runtime type → T4 GPU** before running the next cell. The command below targets CUDA 12. If your runtime uses CUDA 13, use `numba-cuda[cu13]` and the matching CuPy CUDA 13 package instead. See the [Numba CUDA installation guide](https://nvidia.github.io/numba-cuda/user/installation.html).

In [ ]:
!python -m pip install -q "numba-cuda[cu12]" cupy-cuda12x

In [ ]:
import time
import numpy as np
import cupy as cp
from numba import cuda

print("Numba CUDA available:", cuda.is_available())
assert cuda.is_available(), "Enable an NVIDIA GPU runtime before continuing."
print(cuda.get_current_device())

In [ ]:
def ceil_div(a, b):
    return (a + b - 1) // b

def bench_numba(fn, repeats=20):
    """Return median device time in milliseconds; includes a warm-up."""
    fn()
    cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start, end = cuda.event(), cuda.event()
        start.record()
        fn()
        end.record()
        end.synchronize()
        samples.append(cuda.event_elapsed_time(start, end))
    return float(np.median(samples))

def bench_cupy(fn, repeats=20):
    """Return median device time in milliseconds; includes a warm-up."""
    fn()
    cp.cuda.Stream.null.synchronize()
    samples = []
    for _ in range(repeats):
        start, end = cp.cuda.Event(), cp.cuda.Event()
        start.record()
        fn()
        end.record()
        end.synchronize()
        samples.append(cp.cuda.get_elapsed_time(start, end))
    return float(np.median(samples))

## 1. Vector addition — the complete pattern

For `N` elements and `tpb` threads per block, launch `ceil_div(N, tpb)` blocks. The launch may contain a few extra threads, so the bounds check is required.

In [ ]:
@cuda.jit
def vector_add(a, b, out):
    i = cuda.grid(1)
    if i < out.size:
        out[i] = a[i] + b[i]

N = 10_000_000
tpb = 256
bpg = ceil_div(N, tpb)

a_np = np.random.rand(N).astype(np.float32)
b_np = np.random.rand(N).astype(np.float32)
a_d, b_d = cuda.to_device(a_np), cuda.to_device(b_np)
out_d = cuda.device_array(N, dtype=np.float32)

vector_add[bpg, tpb](a_d, b_d, out_d)
result = out_d.copy_to_host()  # synchronous by default
assert np.allclose(result, a_np + b_np)

print(f"Blocks: {bpg:,}")
print(f"Threads launched: {bpg * tpb:,}")
print(f"Extra threads: {bpg * tpb - N:,}")
print(f"Median kernel time: {bench_numba(lambda: vector_add[bpg, tpb](a_d, b_d, out_d)):.3f} ms")

**Questions**

1. Why is `N // tpb` not a safe grid-size calculation in general?
2. Does the launch create exactly `N` threads? Explain.
3. Are all launched threads resident on the GPU at the same instant?

## 2. Element-wise ReLU on AI-shaped data

Complete the kernel. Keep the data as a flat device array while computing, then reshape only the returned host result. Compare with `np.maximum`.

In [ ]:
@cuda.jit
def relu_kernel(x, out):
    i = cuda.grid(1)
    if i < x.size:
        # TODO: write max(0, x[i]) into out[i]
        pass

batch, hidden = 256, 512
x_np = np.random.randn(batch, hidden).astype(np.float32)
x_d = cuda.to_device(x_np.ravel())
out_d = cuda.device_array(x_np.size, dtype=np.float32)
tpb = 256
bpg = ceil_div(x_np.size, tpb)

relu_kernel[bpg, tpb](x_d, out_d)
result = out_d.copy_to_host().reshape(batch, hidden)
assert np.allclose(result, np.maximum(x_np, 0))
print("ReLU verified for", x_np.shape)

## 3. Fuse ReLU + bias add

The two kernels below materialise a full intermediate. Write one fused kernel that computes `max(0, x[i]) + bias[i % bias.size]`, verify it, and compare median times. Do not assume that `N` is a multiple of `bias.size`.

In [ ]:
@cuda.jit
def relu_only(x, out):
    i = cuda.grid(1)
    if i < x.size:
        out[i] = max(0.0, x[i])

@cuda.jit
def bias_add(x, bias, out):
    i = cuda.grid(1)
    if i < x.size:
        out[i] = x[i] + bias[i % bias.size]

@cuda.jit
def fused_relu_bias(x, bias, out):
    i = cuda.grid(1)
    if i < x.size:
        # TODO: compute ReLU and bias addition without a full-size temporary
        pass

N, hidden = 10_000_000, 512
tpb = 256
bpg = ceil_div(N, tpb)
x_np = np.random.randn(N).astype(np.float32)
bias_np = np.random.randn(hidden).astype(np.float32)
x_d, bias_d = cuda.to_device(x_np), cuda.to_device(bias_np)
tmp_d = cuda.device_array(N, dtype=np.float32)
out_d = cuda.device_array(N, dtype=np.float32)

def unfused():
    relu_only[bpg, tpb](x_d, tmp_d)
    bias_add[bpg, tpb](tmp_d, bias_d, out_d)

def fused():
    fused_relu_bias[bpg, tpb](x_d, bias_d, out_d)

In [ ]:
# Verify before comparing performance.
fused()
result = out_d.copy_to_host()
bias_tiled = np.tile(bias_np, ceil_div(N, hidden))[:N]
expected = np.maximum(x_np, 0) + bias_tiled
assert np.allclose(result, expected, atol=1e-5)

t_unfused = bench_numba(unfused)
t_fused = bench_numba(fused)
print(f"Unfused: {t_unfused:.3f} ms")
print(f"Fused:   {t_fused:.3f} ms")
print(f"Ratio:   {t_unfused / t_fused:.2f}x")

**Interpretation:** which full-size array disappears in the fused version? Why is “two kernels become one, so exactly 2× faster” not a valid general prediction?

## 4. Your kernel vs. CuPy

Both implementations use preallocated device outputs. These are **compute-only** timings: host↔device transfers are outside the timed region.

In [ ]:
a_cp, b_cp = cp.asarray(a_np), cp.asarray(b_np)
out_cp = cp.empty_like(a_cp)

# Warm up and verify.
cp.add(a_cp, b_cp, out=out_cp)
assert cp.allclose(out_cp, a_cp + b_cp)

t_cupy = bench_cupy(lambda: cp.add(a_cp, b_cp, out=out_cp))
t_numba = bench_numba(lambda: vector_add[bpg, tpb](a_d, b_d, out_d))

print(f"CuPy:       {t_cupy:.3f} ms")
print(f"Numba CUDA: {t_numba:.3f} ms")
print(f"CuPy time / Numba time: {t_cupy / t_numba:.2f}x")

## 5. Results and conclusion

Fill this table with **your** measurements. Do not copy numbers from another GPU.

| Case | Time (ms) | Verified? | Observation |
|---|---:|:---:|---|
| Vector add — Numba CUDA | | | |
| Vector add — CuPy | | | |
| ReLU + bias — unfused | | | |
| ReLU + bias — fused | | | |

Write one evidence-based sentence: **“My kernel beats / approaches CuPy when … and loses when …”**

## Bonus: 2D matrix transpose

Use `threads = (16, 16)` and map **x → column**, **y → row**. Launch enough blocks in both dimensions and bounds-check both indices.

```python
col, row = cuda.grid(2)
if row < rows and col < cols:
    out[col, row] = x[row, col]
```

Verify against `x_np.T`. Then explain why `cp.transpose(x_cp)` by itself is not equivalent to benchmarking your writing kernel: CuPy's transpose commonly returns a view with changed strides rather than materialising a transposed copy.

## Submission

Submit this notebook with outputs, the completed results table, and your one-sentence conclusion. Include the GPU model shown by the setup cell. Reference solutions for the core examples are in `week08/`; attempt the TODO cells before consulting them.